In [1]:
!pip install datasets pandas tqdm

from datasets import load_dataset
import pandas as pd
import json
import re
from tqdm import tqdm

# Step 1: Load the dataset from HuggingFace
dataset = load_dataset("tatsu-lab/alpaca")

# Step 2: Convert to pandas DataFrame
df = pd.DataFrame(dataset["train"])
print(f"✅ Loaded {len(df)} records from the Alpaca dataset")
print(df.head())

# Step 3: Clean text
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    return text

for col in ["instruction", "input", "output"]:
    df[col] = df[col].apply(clean_text)

# Step 4: Remove empty or short rows
df = df[df["instruction"].str.len() > 5]
df = df[df["output"].str.len() > 5]
df.reset_index(drop=True, inplace=True)
print(f"✅ Cleaned dataset has {len(df)} usable rows")

# Step 5: Save as JSONL for instruction tuning
output_file = "cleaned_alpaca_dataset.jsonl"
with open(output_file, "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        json.dump({
            "instruction": row["instruction"],
            "input": row["input"],
            "output": row["output"]
        }, f)
        f.write("\n")

print(f"✅ Cleaned and formatted dataset saved as {output_file}")

# Step 6: Preview some random examples
df.sample(5)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

✅ Loaded 52002 records from the Alpaca dataset
                                         instruction input  \
0               Give three tips for staying healthy.         
1                 What are the three primary colors?         
2                 Describe the structure of an atom.         
3                   How can we reduce air pollution?         
4  Describe a time when you had to make a difficu...         

                                              output  \
0  1.Eat a balanced diet and make sure to include...   
1  The three primary colors are red, blue, and ye...   
2  An atom is made up of a nucleus, which contain...   
3  There are a number of ways to reduce air pollu...   
4  I had to make a difficult decision when I was ...   

                                                text  
0  Below is an instruction that describes a task....  
1  Below is an instruction that describes a task....  
2  Below is an instruction that describes a task....  
3  Below is an instruct

,instruction,input,output,text
5117,Generates a report on the size of the populati...,,The population of Venezuela is estimated to be...,Below is an instruction that describes a task....
44218,"Given a sentence, replace the pronoun with the...","The man had the idea, but he didn't follow thr...","The man had the idea, but he didn't follow thr...","Below is an instruction that describes a task,..."
32134,Write a 2-3 sentence description of a stranger,,"He was a tall, thin man with a thin mustache a...",Below is an instruction that describes a task....
35140,Summarize the primary differences between the ...,,The primary differences between the US and Can...,Below is an instruction that describes a task....
5398,Classify the sentiment of this statement,Chuck's performance on the project was exemplary.,Positive,"Below is an instruction that describes a task,..."
